In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ==============================================================================
# 关键：导入需要的库
# ==============================================================================
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from medpy.metric.binary import hd95

# ==============================================================================
# --- 1. 配置 (已修改为读取指定的测试集文件) ---
# ==============================================================================
# Hydra 配置名和官方预训练权重路径
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2/checkpoints/sam2.1_hiera_base_plus.pt"

# !! 关键：指定数据集根目录和要使用的测试集文件 !!
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/BUSI_for_SAM2"
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets", "val.txt")

# ==============================================================================
# --- 2. 初始化SAM2模型和预测器 ---
# ==============================================================================
print("Loading official pre-trained SAM2 model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam2(SAM2_CFG_NAME, SAM2_CHECKPOINT_PATH, device=device)
predictor = SAM2ImagePredictor(model)
print("Model and Predictor loaded.")

# ==============================================================================
# --- 3. 定义指标计算函数 (保持不变) ---
# ==============================================================================
def calculate_metrics(gt_mask, pred_mask):
    """计算Dice, IoU, 和 HD95"""
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0
    
    if not np.any(gt_mask_bool) and not np.any(pred_mask_bool):
        dice, iou = 1.0, 1.0
    elif not np.any(gt_mask_bool) or not np.any(pred_mask_bool):
        dice, iou = 0.0, 0.0
    else:
        tp = np.logical_and(pred_mask_bool, gt_mask_bool).sum()
        fp = np.logical_and(pred_mask_bool, ~gt_mask_bool).sum()
        fn = np.logical_and(~pred_mask_bool, gt_mask_bool).sum()
        dice = (2. * tp) / (2 * tp + fp + fn + 1e-8)
        iou = tp / (tp + fp + fn + 1e-8)
    
    hd95_score = np.nan
    if np.any(gt_mask_bool) and np.any(pred_mask_bool):
        try:
            hd95_score = hd95(pred_mask_bool, gt_mask_bool)
        except RuntimeError: pass
            
    return dice, iou, hd95_score

# ==============================================================================
# --- !! 核心修改: 从指定的文本文件读取测试样本 !! ---
# ==============================================================================
results = []

# --- 从指定的测试集文件中读取样本列表 ---
if not os.path.exists(TEST_SET_FILE):
    raise FileNotFoundError(f"Test set file not found: {TEST_SET_FILE}")

with open(TEST_SET_FILE, 'r') as f:
    # 假设文件内容是样本名，每行一个，例如 cju0, cju1...
    test_sample_names = [line.strip() for line in f.readlines()]

print(f"\nFound {len(test_sample_names)} samples in '{os.path.basename(TEST_SET_FILE)}'. Starting evaluation...")

# 定义图像和掩码的文件夹路径
images_dir = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
annotations_dir = os.path.join(SPLIT_DATASET_ROOT, "Annotations")

# ==============================================================================
# --- 5. 遍历、分割和评估 (现在遍历从文件读取的 test_sample_names) ---
# ==============================================================================
for sample_name in tqdm(test_sample_names, desc="Evaluating Default SAM2 on Test Set"):
    # 构建完整的文件路径
    # 这种 _for_SAM2 的数据集结构通常是：JPEGImages/{sample_name}/00000.jpg
    image_path = os.path.join(images_dir, sample_name, "00000.jpg")
    mask_path = os.path.join(annotations_dir, sample_name, "00000.png")
    
    # 假设 CVC-ClinicDB 数据集只有一个类别 'polyp'
    category = 'polyp'
    filename = os.path.basename(image_path)

    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"Warning: Files for sample '{sample_name}' not found. Skipping.")
        continue

    # --- 后续的图像读取和推理逻辑保持不变 ---
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    predictor.set_image(image_rgb)
    contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: continue
    
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    box_prompt = np.array([[x, y, x + w, y + h]])

    masks, scores, logits = predictor.predict(box=box_prompt, multimask_output=False)
    
    pred_mask = (masks[0] * 255).astype(np.uint8)
    
    dice, iou, hd95_score = calculate_metrics(gt_mask, pred_mask)
    
    results.append({
        "category": category, "filename": filename, 
        "dice": dice, "iou": iou, "hd95": hd95_score
    })
    
    predictor.reset_predictor()

# ==============================================================================
# --- 6. 最终报告 (保持不变) ---
# ==============================================================================
if results:
    df = pd.DataFrame(results)
    
    agg_funcs = {
        'dice': ['mean', 'std'], 'iou': ['mean', 'std'],
        'hd95': ['mean', 'std'], 'filename': ['count']
    }
    category_summary = df.groupby('category').agg(agg_funcs).reset_index()
    
    category_summary.columns = ['_'.join(col).strip() for col in category_summary.columns.values]
    category_summary.rename(columns={'category_': 'Category', 'filename_count': 'Count'}, inplace=True)

    overall_metrics = {
        'Category': 'Overall', 'dice_mean': df['dice'].mean(), 'dice_std': df['dice'].std(),
        'iou_mean': df['iou'].mean(), 'iou_std': df['iou'].std(),
        'hd95_mean': df['hd95'].mean(), 'hd95_std': df['hd95'].std(), 'Count': len(df)
    }
    
    overall_df = pd.DataFrame([overall_metrics])
    final_summary = pd.concat([category_summary, overall_df], ignore_index=True)

    print("\n" + "="*85)
    print(f"--- Evaluation Summary for Default SAM2 (on {os.path.basename(TEST_SET_FILE)}) ---")
    print("="*85)
    
    formatters = {
        'dice_mean': "{:.4f}".format, 'dice_std': "{:.4f}".format,
        'iou_mean': "{:.4f}".format, 'iou_std': "{:.4f}".format,
        'hd95_mean': "{:.2f}".format, 'hd95_std': "{:.2f}".format,
    }
    print(final_summary.to_string(index=False, formatters=formatters))
    print("="*85)

else:
    print("No images were processed.")

Loading official pre-trained SAM2 model...
Model and Predictor loaded.

Found 130 samples in 'val.txt'. Starting evaluation...


Evaluating Default SAM2 on Test Set: 100%|██████████| 130/130 [00:18<00:00,  7.20it/s]


--- Evaluation Summary for Default SAM2 (on val.txt) ---
Category dice_mean dice_std iou_mean iou_std hd95_mean hd95_std  Count
   polyp    0.8993   0.0762   0.8238  0.1005     20.05    20.21    130
 Overall    0.8993   0.0762   0.8238  0.1005     20.05    20.21    130
